In [1]:
import timm, torch

BACKBONE = {
    "EfficientNet-B0": "efficientnet_b0",
    "ResNet50":        "resnet50",
    "DINOv2-S":        "vit_small_patch14_dinov2.lvd142m",
    "DINOv2-B":        "vit_base_patch14_dinov2.lvd142m",
}
DATA = {"v1": (3, 16), "v2": (5, 16), "v3": (5, 32)}   # (jumlah view, jumlah irisan per view)

def backbone_kwargs(name, img_size=224):              # sama dengan script pelatihan
    if "dinov2" in name:
        return dict(img_size=img_size, drop_path_rate=0.05)
    return {}

for nama, bb in BACKBONE.items():
    enc = timm.create_model(bb, pretrained=False, num_classes=0, in_chans=3, **backbone_kwargs(bb)).eval()
    f = enc.num_features
    with torch.no_grad():
        out = enc(torch.zeros(48, 3, 224, 224))           # 48 irisan = 1 studi data v1 (3 view x 16 irisan)
    n_enc = sum(p.numel() for p in enc.parameters())
    n_total_3v = n_enc + 3 * (f * 128 + 128 + 128 + 1) + (f * 3 * 12 + 12)   # + attention 3 view + kepala
    print(f"\n{nama} ({bb})")
    print(f"  fitur per irisan (num_features)      : {f}")
    print(f"  keluaran encoder untuk 1 studi v1    : {tuple(out.shape)}")
    for v, (V, D) in DATA.items():
        print(f"  data {v}: {V * D:3d} vektor fitur per studi | masukan kepala {V} view = {f * V}")
    print(f"  parameter KneeNet 3 view             : {n_total_3v / 1e6:.1f} juta")


EfficientNet-B0 (efficientnet_b0)
  fitur per irisan (num_features)      : 1280
  keluaran encoder untuk 1 studi v1    : (48, 1280)
  data v1:  48 vektor fitur per studi | masukan kepala 3 view = 3840
  data v2:  80 vektor fitur per studi | masukan kepala 5 view = 6400
  data v3: 160 vektor fitur per studi | masukan kepala 5 view = 6400
  parameter KneeNet 3 view             : 4.5 juta

ResNet50 (resnet50)
  fitur per irisan (num_features)      : 2048
  keluaran encoder untuk 1 studi v1    : (48, 2048)
  data v1:  48 vektor fitur per studi | masukan kepala 3 view = 6144
  data v2:  80 vektor fitur per studi | masukan kepala 5 view = 10240
  data v3: 160 vektor fitur per studi | masukan kepala 5 view = 10240
  parameter KneeNet 3 view             : 24.4 juta

DINOv2-S (vit_small_patch14_dinov2.lvd142m)
  fitur per irisan (num_features)      : 384
  keluaran encoder untuk 1 studi v1    : (48, 384)
  data v1:  48 vektor fitur per studi | masukan kepala 3 view = 1152
  data v2:  80 vektor